# NeuroBridge-S4 Graph Learning — Phase 3: Biological Adaptation Graph Construction

**Repository:** NeuroBridge-S4-Graph-Learning  
**Phase:** 3 of 5  
**Date:** June 2026

---

## Purpose

This notebook converts each pseudo-crew participant into a **biological adaptation graph**.

Each graph shows biological domains as nodes and conceptual or co-activation relationships
as edges. This is the first computational step from tabular representation toward
graph learning.

## Why this matters

A table splits a participant into columns — isolated measurements with no structural
relationships between them.

A graph **reassembles the participant as a connected biological system**.

This structure makes later graph features, embeddings, novelty detection, and
explainable subgraph profiles computationally possible.

---

## Scientific guardrails

- This is **not** actual Artemis II data. These are processed **proxy-data outputs**
  from the NeuroBridge-S4 methodology applied to NHANES public data.
- Graphs are **research interpretation artifacts**, not clinical records.
- Graphs do **not diagnose** medical or psychological states.
- **Edges** are conceptual or co-activation relationships — **not causal proof**.
- The method is intended to help identify which biological systems should be
  **reviewed more closely** by qualified researchers.

## 0. Setup

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, HTML, FileLink

warnings.filterwarnings('ignore', category=UserWarning)

# ── path setup ──────────────────────────────────────────────────────────────
repo_root = Path().resolve().parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from neurobridge_graph.graph_builder import (
    build_all_subject_graphs,
    export_node_table,
    export_edge_table,
    export_graph_summary,
    save_graphml_files,
    GUARDRAIL,
)
from neurobridge_graph.visualization import draw_subject_graph, draw_graph_summary_bar
from neurobridge_graph.interactive import (
    export_all_graphs_html,
    export_index_html,
)

# ── output directories ───────────────────────────────────────────────────────
TABLES_DIR   = repo_root / 'results' / 'tables'
GRAPHS_DIR   = repo_root / 'results' / 'graphs'
FIGURES_DIR  = repo_root / 'results' / 'figures'
HTML_DIR     = repo_root / 'results' / 'html'
REPORTS_DIR  = repo_root / 'results' / 'reports'
DATA_DIR     = repo_root / 'data' / 'processed'

for d in [TABLES_DIR, GRAPHS_DIR, FIGURES_DIR, HTML_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Setup complete.')
print(f'  repo root : {repo_root}')
print(f'  data dir  : {DATA_DIR}')

Setup complete.
  repo root : /Users/rm/Desktop/NeuroBridge-S4-Graph-Learning
  data dir  : /Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/data/processed


## 1. Load Phase 2 Processed Outputs

In [2]:
from neurobridge_graph.data_loader import (
    load_domain_scores,
    load_baci_scores,
    load_pseudo_crew,
    load_deviation_scores,
    load_baci_sensitivity,
    load_crew_level_summary,
)

# ── required files ───────────────────────────────────────────────────────────
REQUIRED = {
    'pseudo_crew.csv':   load_pseudo_crew,
    'domain_scores.csv': load_domain_scores,
    'baci_scores.csv':   load_baci_scores,
}

# ── optional files ───────────────────────────────────────────────────────────
OPTIONAL = {
    'deviation_scores.csv':  load_deviation_scores,
    'baci_sensitivity.csv':  load_baci_sensitivity,
    'crew_level_summary.csv': load_crew_level_summary,
}

readiness_rows = []
loaded = {}

for fname, loader in {**REQUIRED, **OPTIONAL}.items():
    req = 'required' if fname in REQUIRED else 'optional'
    fpath = DATA_DIR / fname
    try:
        df = loader()
        loaded[fname] = df
        readiness_rows.append({
            'file': fname, 'required_or_optional': req,
            'status': '✓ loaded', 'rows': len(df),
            'columns': len(df.columns), 'notes': '',
        })
    except FileNotFoundError as e:
        loaded[fname] = None
        readiness_rows.append({
            'file': fname, 'required_or_optional': req,
            'status': '✗ missing', 'rows': '', 'columns': '',
            'notes': 'file not found',
        })
        if req == 'required':
            raise

readiness_df = pd.DataFrame(readiness_rows)
readiness_df.to_csv(TABLES_DIR / 'phase2_readiness_check.csv', index=False)
display(readiness_df.style.hide(axis='index'))
print(f'\nSaved → {TABLES_DIR / "phase2_readiness_check.csv"}')

file,required_or_optional,status,rows,columns,notes
pseudo_crew.csv,required,✓ loaded,4,9,
domain_scores.csv,required,✓ loaded,4,6,
baci_scores.csv,required,✓ loaded,4,4,
deviation_scores.csv,optional,✓ loaded,4,20,
baci_sensitivity.csv,optional,✓ loaded,12,4,
crew_level_summary.csv,optional,✓ loaded,4,7,



Saved → /Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/results/tables/phase2_readiness_check.csv


In [3]:
# Assign to named variables for convenience
domain_scores = loaded['domain_scores.csv']
baci_scores   = loaded['baci_scores.csv']
pseudo_crew   = loaded['pseudo_crew.csv']

deviation_scores  = loaded.get('deviation_scores.csv')
baci_sensitivity  = loaded.get('baci_sensitivity.csv')
crew_summary      = loaded.get('crew_level_summary.csv')

print('domain_scores shape:', domain_scores.shape)
print('domain columns:', list(domain_scores.columns))
print()
display(domain_scores)

domain_scores shape: (4, 6)
domain columns: ['Cardiovascular regulation', 'Metabolic regulation', 'Body composition / physical status', 'Inflammation / immune-adjacent', 'Hematologic / oxygen-carrying', 'Recovery-related markers']



,Cardiovascular regulation,Metabolic regulation,Body composition / physical status,Inflammation / immune-adjacent,Hematologic / oxygen-carrying,Recovery-related markers
Crew 97774,0.483921,0.485515,1.867267,0.422395,2.107285,0.943802
Crew 94465,0.948613,0.835169,1.322530,0.496448,0.627734,0.806384
Crew 99813,0.193564,1.157896,0.194916,1.098454,0.701952,0.836982
Crew 100987,1.881908,0.689183,0.668464,0.486165,0.614935,0.705149


## 2. Graph Schema Explanation

### Nodes
Each **node** represents one **biological domain** present in the `domain_scores.csv` file.
The domain score is the mean absolute z-score deviation of biomarkers in that domain,
relative to the NHANES reference population.

### Node attributes
| Attribute | Meaning |
|---|---|
| `domain_score` | Value from domain_scores.csv |
| `activation` | Absolute magnitude of domain score |
| `activation_level` | low / mild / moderate / high |
| `node_type` | `biological_domain` |
| `data_source` | Processed NeuroBridge-S4 proxy outputs |
| `interpretation` | Plain-language phrase |

### Activation thresholds (mean \|z\|)
| Level | Range |
|---|---|
| low | < 0.75 |
| mild | 0.75 – 1.0 |
| moderate | 1.0 – 1.5 |
| high | ≥ 1.5 |

### Edge types
| Type | Meaning |
|---|---|
| `conceptual_biological_relationship` | Domain pair connected by known biological coupling |
| `within_subject_coactivation` | Both domains show activation ≥ threshold for this subject |

### Graph metadata
Each graph carries: `subject_id`, `baci_score`, `baci_category`,
`n_active_domains`, `top_domain`, `graph_type`, `guardrail`.

### Why this design is future-proof
Domain nodes can receive additional features in Phase 4 (graph feature extraction).
The graph structure is directly usable by PyTorch Geometric / DGL for GNN experiments.
Future edge types (`observed_reference_relationship`, `decision_support_relationship`)
are documented in the schema and can be added without restructuring.

## 3. Build Subject-Level Graphs

In [4]:
graphs = build_all_subject_graphs(
    domain_scores=domain_scores,
    baci_df=baci_scores,
    add_coactivation_edges=True,
    activation_threshold=1.0,
)

print(f'Built {len(graphs)} subject graph(s):')
for sid, G in graphs.items():
    print(
        f'  Subject {sid}: '
        f'{G.number_of_nodes()} nodes, '
        f'{G.number_of_edges()} edges | '
        f'BACI={G.graph.get("baci_score")} ({G.graph.get("baci_category")}) | '
        f'top domain: {G.graph.get("top_domain")}'
    )

Built 4 subject graph(s):
  Subject Crew 97774: 6 nodes, 5 edges | BACI=41.5 (mild coherence) | top domain: Hematologic / oxygen-carrying
  Subject Crew 94465: 6 nodes, 4 edges | BACI=22.1 (low coherence) | top domain: Body composition / physical status
  Subject Crew 99813: 6 nodes, 4 edges | BACI=31.5 (mild coherence) | top domain: Metabolic regulation
  Subject Crew 100987: 6 nodes, 4 edges | BACI=28.6 (mild coherence) | top domain: Cardiovascular regulation


In [5]:
# Save GraphML files
graphml_paths = save_graphml_files(graphs, GRAPHS_DIR)
print('Saved GraphML files:')
for p in graphml_paths:
    print(f'  {p.relative_to(repo_root)}')

Saved GraphML files:
  results/graphs/subject_graph_Crew_97774.graphml
  results/graphs/subject_graph_Crew_94465.graphml
  results/graphs/subject_graph_Crew_99813.graphml
  results/graphs/subject_graph_Crew_100987.graphml


## 4. Export Node / Edge / Summary Tables

In [6]:
node_table    = export_node_table(graphs)
edge_table    = export_edge_table(graphs)
summary_table = export_graph_summary(graphs)

node_table.to_csv(TABLES_DIR / 'subject_nodes.csv', index=False)
edge_table.to_csv(TABLES_DIR / 'subject_edges.csv', index=False)
summary_table.to_csv(TABLES_DIR / 'phase3_graph_summary.csv', index=False)

print('Node table:')
display(node_table[['graph_subject_id','node','domain_score','activation','activation_level','interpretation']].head(10))

print('\nEdge table:')
display(edge_table[['graph_subject_id','source_node','target_node','edge_type','weight']].head(10))

print('\nGraph summary:')
display(summary_table)

Node table:


,graph_subject_id,node,domain_score,activation,activation_level,interpretation
0,Crew 97774,Cardiovascular regulation,0.483921,0.483921,low,Cardiovascular regulation: reference-relative ...
1,Crew 97774,Metabolic regulation,0.485515,0.485515,low,Metabolic regulation: reference-relative signa...
2,Crew 97774,Body composition / physical status,1.867267,1.867267,high,Body composition / physical status: high domai...
3,Crew 97774,Inflammation / immune-adjacent,0.422395,0.422395,low,Inflammation / immune-adjacent: reference-rela...
4,Crew 97774,Hematologic / oxygen-carrying,2.107285,2.107285,high,Hematologic / oxygen-carrying: high domain act...
5,Crew 97774,Recovery-related markers,0.943802,0.943802,mild,Recovery-related markers: mild reference-relat...
6,Crew 94465,Cardiovascular regulation,0.948613,0.948613,mild,Cardiovascular regulation: mild reference-rela...
7,Crew 94465,Metabolic regulation,0.835169,0.835169,mild,Metabolic regulation: mild reference-relative ...
8,Crew 94465,Body composition / physical status,1.322530,1.322530,moderate,Body composition / physical status: moderate d...
9,Crew 94465,Inflammation / immune-adjacent,0.496448,0.496448,low,Inflammation / immune-adjacent: reference-rela...



Edge table:


,graph_subject_id,source_node,target_node,edge_type,weight
0,Crew 97774,Cardiovascular regulation,Metabolic regulation,conceptual_biological_relationship,1.000000
1,Crew 97774,Cardiovascular regulation,Hematologic / oxygen-carrying,conceptual_biological_relationship,1.000000
2,Crew 97774,Metabolic regulation,Body composition / physical status,conceptual_biological_relationship,1.000000
3,Crew 97774,Metabolic regulation,Inflammation / immune-adjacent,conceptual_biological_relationship,1.000000
4,Crew 97774,Body composition / physical status,Hematologic / oxygen-carrying,within_subject_coactivation,1.987276
5,Crew 94465,Cardiovascular regulation,Metabolic regulation,conceptual_biological_relationship,1.000000
6,Crew 94465,Cardiovascular regulation,Hematologic / oxygen-carrying,conceptual_biological_relationship,1.000000
7,Crew 94465,Metabolic regulation,Body composition / physical status,conceptual_biological_relationship,1.000000
8,Crew 94465,Metabolic regulation,Inflammation / immune-adjacent,conceptual_biological_relationship,1.000000
9,Crew 99813,Cardiovascular regulation,Metabolic regulation,conceptual_biological_relationship,1.000000



Graph summary:


,subject_id,n_nodes,n_edges,baci_score,baci_category,n_domains,n_active_domains,max_domain_activation,top_domain,graph_type,source_project,guardrail
0,Crew 97774,6,5,41.5,mild coherence,6,2,2.107285,Hematologic / oxygen-carrying,subject_level_biological_adaptation_graph,NeuroBridge-S4 Graph Learning,Research interpretation only; not diagnosis or...
1,Crew 94465,6,4,22.1,low coherence,6,1,1.322530,Body composition / physical status,subject_level_biological_adaptation_graph,NeuroBridge-S4 Graph Learning,Research interpretation only; not diagnosis or...
2,Crew 99813,6,4,31.5,mild coherence,6,2,1.157896,Metabolic regulation,subject_level_biological_adaptation_graph,NeuroBridge-S4 Graph Learning,Research interpretation only; not diagnosis or...
3,Crew 100987,6,4,28.6,mild coherence,6,1,1.881908,Cardiovascular regulation,subject_level_biological_adaptation_graph,NeuroBridge-S4 Graph Learning,Research interpretation only; not diagnosis or...


## 5. Static Graph Visualizations

In [7]:
import re

for sid, G in graphs.items():
    safe_id = re.sub(r'[^\w\-]', '_', str(sid))
    png_path = FIGURES_DIR / f'subject_graph_{safe_id}.png'
    fig, ax = draw_subject_graph(
        G,
        output_path=png_path,
        show=False,
    )
    print(f'Subject {sid} — saved PNG: {png_path.relative_to(repo_root)}')
    display(fig)
    plt.close(fig)

    display(HTML(f'''
    <div style="background:#f8f9fa;border-left:4px solid #AED6F1;
                padding:10px 16px;margin:8px 0;font-size:0.9em;">
      <b>Reviewer note:</b><br>
      • This graph shows the participant as a connected biological system.<br>
      • Larger nodes indicate stronger domain activation.<br>
      • Edges indicate conceptual relationships or within-subject co-activation,
        not causal proof.<br>
      • This helps identify which biological domains should be reviewed more closely.
    </div>
    '''))

Subject Crew 97774 — saved PNG: results/figures/subject_graph_Crew_97774.png


<Figure size 1000x700 with 1 Axes>

Subject Crew 94465 — saved PNG: results/figures/subject_graph_Crew_94465.png


<Figure size 1000x700 with 1 Axes>

Subject Crew 99813 — saved PNG: results/figures/subject_graph_Crew_99813.png


<Figure size 1000x700 with 1 Axes>

Subject Crew 100987 — saved PNG: results/figures/subject_graph_Crew_100987.png


<Figure size 1000x700 with 1 Axes>

## 6. Interactive HTML Graph Visualizations

Interactive HTML graphs allow reviewers to inspect node and edge attributes
without reading code. Nodes are draggable; hover to see domain scores,
activation levels, and interpretation text.

In [8]:
html_paths = export_all_graphs_html(graphs, HTML_DIR)
index_path = export_index_html(html_paths, HTML_DIR / 'index.html', graphs=graphs)

print('Interactive HTML graphs saved:')
for p in html_paths:
    print(f'  {p.relative_to(repo_root)}')
print(f'\nIndex page: {index_path.relative_to(repo_root)}')

Interactive HTML graphs saved:
  results/html/subject_graph_Crew_97774.html
  results/html/subject_graph_Crew_94465.html
  results/html/subject_graph_Crew_99813.html
  results/html/subject_graph_Crew_100987.html

Index page: results/html/index.html


In [9]:
# Show clickable links in the notebook
display(HTML('<h4>Interactive graph links (open in browser):</h4>'))
for p in [index_path] + html_paths:
    display(FileLink(str(p.relative_to(repo_root))))

# Embed first graph inline (if reasonably small)
if html_paths:
    first_html = html_paths[0].read_text(encoding='utf-8')
    if len(first_html) < 3_000_000:
        from IPython.display import IFrame
        display(HTML('<h4>Embedded interactive graph (first subject):</h4>'))
        display(IFrame(src=str(html_paths[0].relative_to(repo_root)), width='100%', height='620px'))

/Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/notebooks/results/html/index.html

/Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/notebooks/results/html/subject_graph_Crew_97774.html

/Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/notebooks/results/html/subject_graph_Crew_94465.html

/Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/notebooks/results/html/subject_graph_Crew_99813.html

/Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/notebooks/results/html/subject_graph_Crew_100987.html

## 7. Compare Graph Summaries

In [10]:
# Summary table
cols_to_show = [
    'subject_id','n_nodes','n_edges','n_active_domains',
    'top_domain','max_domain_activation','baci_score','baci_category'
]
available_cols = [c for c in cols_to_show if c in summary_table.columns]
display(summary_table[available_cols])

,subject_id,n_nodes,n_edges,n_active_domains,top_domain,max_domain_activation,baci_score,baci_category
0,Crew 97774,6,5,2,Hematologic / oxygen-carrying,2.107285,41.5,mild coherence
1,Crew 94465,6,4,1,Body composition / physical status,1.322530,22.1,low coherence
2,Crew 99813,6,4,2,Metabolic regulation,1.157896,31.5,mild coherence
3,Crew 100987,6,4,1,Cardiovascular regulation,1.881908,28.6,mild coherence


In [11]:
# Bar chart
bar_path = FIGURES_DIR / 'phase3_graph_summary.png'
fig, axes = draw_graph_summary_bar(summary_table, output_path=bar_path, show=False)
display(fig)
plt.close(fig)
print(f'Saved → {bar_path.relative_to(repo_root)}')

<Figure size 1200x400 with 2 Axes>

Saved → results/figures/phase3_graph_summary.png


## 8. Plain-Language Phase 3 Report

In [12]:
from datetime import date

n_graphs = len(graphs)

# Highest activation subject
top_act_row = summary_table.loc[summary_table['max_domain_activation'].astype(float).idxmax()]
# Highest BACI subject
baci_vals = summary_table['baci_score'].apply(
    lambda x: float(x) if x not in (None, 'None', '') else float('-inf')
)
top_baci_row = summary_table.loc[baci_vals.idxmax()]

lines = []
lines.append('=' * 70)
lines.append('PHASE 3 REPORT — NeuroBridge-S4 Biological Adaptation Graphs')
lines.append(f'Generated: {date.today()}')
lines.append('=' * 70)
lines.append('')
lines.append(f'Graphs built: {n_graphs}')
lines.append('')
lines.append('Per-participant summary:')
for _, row in summary_table.iterrows():
    lines.append(
        f'  Subject {row["subject_id"]}: '
        f'top domain = {row.get("top_domain","n/a")}, '
        f'BACI = {row.get("baci_score","n/a")} ({row.get("baci_category","n/a")}), '
        f'active domains = {row.get("n_active_domains","n/a")}'
    )
lines.append('')
lines.append(
    f'Highest activation: Subject {top_act_row["subject_id"]} '
    f'(max domain activation = {top_act_row["max_domain_activation"]})'
)
lines.append(
    f'Highest BACI: Subject {top_baci_row["subject_id"]} '
    f'(BACI = {top_baci_row.get("baci_score","n/a")})'
)
lines.append('')
lines.append('What Phase 3 adds to NeuroBridge-S4:')
lines.append(
    '  The original NeuroBridge-S4 project produced per-participant scores\n'
    '  (domain scores, BACI, deviation vectors) as tabular outputs.\n'
    '  Phase 3 converts those tables into structured NetworkX graphs,\n'
    '  where each participant is represented as a connected biological system.\n'
    '  This enables downstream graph feature extraction, graph embeddings,\n'
    '  novelty detection, and eventually graph neural network experiments.'
)
lines.append('')
lines.append('Output locations:')
lines.append(f'  Static PNG graphs : results/figures/')
lines.append(f'  GraphML files     : results/graphs/')
lines.append(f'  Interactive HTML  : results/html/  (open index.html)')
lines.append(f'  Node table        : results/tables/subject_nodes.csv')
lines.append(f'  Edge table        : results/tables/subject_edges.csv')
lines.append(f'  Summary table     : results/tables/phase3_graph_summary.csv')
lines.append('')
lines.append('-' * 70)
lines.append(GUARDRAIL)
lines.append('-' * 70)

report_text = '\n'.join(lines)
report_path = REPORTS_DIR / 'phase3_subject_graph_report.txt'
report_path.write_text(report_text, encoding='utf-8')

print(report_text)
print(f'\nSaved → {report_path.relative_to(repo_root)}')

PHASE 3 REPORT — NeuroBridge-S4 Biological Adaptation Graphs
Generated: 2026-06-06

Graphs built: 4

Per-participant summary:
  Subject Crew 97774: top domain = Hematologic / oxygen-carrying, BACI = 41.5 (mild coherence), active domains = 2
  Subject Crew 94465: top domain = Body composition / physical status, BACI = 22.1 (low coherence), active domains = 1
  Subject Crew 99813: top domain = Metabolic regulation, BACI = 31.5 (mild coherence), active domains = 2
  Subject Crew 100987: top domain = Cardiovascular regulation, BACI = 28.6 (mild coherence), active domains = 1

Highest activation: Subject Crew 97774 (max domain activation = 2.107285)
Highest BACI: Subject Crew 97774 (BACI = 41.5)

What Phase 3 adds to NeuroBridge-S4:
  The original NeuroBridge-S4 project produced per-participant scores
  (domain scores, BACI, deviation vectors) as tabular outputs.
  Phase 3 converts those tables into structured NetworkX graphs,
  where each participant is represented as a connected biologica

## 9. Conclusion

Phase 3 has created the computational graph objects needed for later phases.

Each pseudo-crew participant is now represented as a **biological adaptation graph**
with:
- domain nodes annotated with activation levels and interpretation text;
- conceptual edges capturing known biological coupling;
- co-activation edges flagging domains that are simultaneously elevated;
- graph-level metadata including BACI score, top domain, and guardrail text.

### Next phases

| Phase | Goal |
|---|---|
| Phase 4 | Compute graph-level and node-level features (centrality, active subgraph sizes, coherence-weighted scores) |
| Phase 5 | Compute graph embeddings, PCA, cosine similarity, distance-to-reference |

---

*Research interpretation only. Not diagnosis or treatment guidance.*